# Vinamilk embedding benchmark (full public corpus, cloud GPU)

This notebook compares the proposal's exact embedding models on all 4,750 public manifest passages and the same 12 role-labeled business cases. It reads a committed JSONL export derived from ready database chunks, restricted to document IDs in `data/vietnam_enterprises/manifest.json`; it does not connect to Supabase or need `DATABASE_URL`. The corpus retains document role labels and the evaluator applies them before scoring.

In Kaggle settings, enable Internet and a GPU accelerator. Add a Kaggle Secret named `GITHUB_READ_TOKEN` with read-only Contents access to this private repository. Never paste the token into a cell. Results are smoke comparisons: 12 queries and document-level labels are still too few to select a production model.

In [ ]:
from pathlib import Path
from kaggle_secrets import UserSecretsClient
import os, subprocess, sys

REPOSITORY = "Lenhan231/enterprise-knowledge-ai-agent"
BRANCH = "feature/vietnam-enterprise-corpus"
repo = Path("/kaggle/working/enterprise-knowledge-ai-agent")
token = UserSecretsClient().get_secret("GITHUB_READ_TOKEN")
askpass = Path("/kaggle/working/git-askpass.sh")
askpass.write_text('#!/bin/sh\ncase "$1" in *Username*) echo x-access-token ;; *Password*) echo "$GITHUB_READ_TOKEN" ;; esac\n')
askpass.chmod(0o700)
env = os.environ | {"GITHUB_READ_TOKEN": token, "GIT_ASKPASS": str(askpass), "GIT_TERMINAL_PROMPT": "0"}
subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH, f"https://github.com/{REPOSITORY}.git", str(repo)], env=env, check=True)
del token, env
askpass.unlink(missing_ok=True)
%cd /kaggle/working/enterprise-knowledge-ai-agent

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "sentence-transformers==6.1.0"], check=True)
import torch
subprocess.run(["nvidia-smi"], check=False)
print("torch", torch.__version__, "CUDA", torch.cuda.is_available(), "GPU", torch.cuda.get_device_name(0) if torch.cuda.is_available() else None)
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime was requested but CUDA is unavailable. Restart with a Kaggle GPU accelerator.")
import os
os.environ["HF_HOME"] = "/kaggle/working/model-cache/huggingface"

In [ ]:
corpus = Path("data/evaluation/vinamilk_public_embedding_corpus.jsonl")
cases = Path("data/evaluation/vinamilk_retrieval_cases.json")
assert corpus.is_file() and cases.is_file()
import hashlib, json
metadata = json.loads(Path(str(corpus) + ".metadata.json").read_text())
assert hashlib.sha256(corpus.read_bytes()).hexdigest() == metadata["corpus_sha256"]
print(metadata)

In [ ]:
# Run each model in a fresh process so GPU allocations are released between models.
models = ["bge-m3", "qwen3-embedding", "e5-large-v2"]
results = Path("/kaggle/working/embedding-results")
results.mkdir(exist_ok=True)
for model in models:
    report = results / f"vinamilk-embedding-{model}-full-public-corpus.json"
    command = [sys.executable, "scripts/evaluate_embeddings.py", "--model", model, "--corpus", str(corpus), "--cases", str(cases), "--device", "cuda", "--batch-size", "4", "--max-chunks-per-document", "1000", "--top-k", "30", "--output", str(report)]
    subprocess.run(command, check=True)
    print(f"Saved {report}")

## Results

Download the three JSON reports from `/kaggle/working/embedding-results/`. Each report contains model revision, case/corpus hashes, GPU name, timings, peak RAM, role-filtered rankings and aggregate/per-case metrics. Keep the result JSON with the software environment details when importing it into `artifacts/evaluation/`.